# Ex0 Intro to GNSS Navigation

---

1. [Installation](#1-installation)
2. [Imports](#2-imports)
3. [Load Data from GitHub](#3-load-data-from-github)
4. [Constants & Preprocessing](#4-constants--preprocessing)
5. [Satellite Position Functions](#5-satellite-position-functions)
6. [Pseudorange Correction](#6-pseudorange-correction)
7. [Positioning & Coordinate Utilities](#7-positioning--coordinate-utilities)
8. [Trajectory Computation](#8-trajectory-computation)
9. [Export Results](#9-export-results)

---

## 1. Installation
Install required libraries if not already available.

In [ ]:
!pip install georinex

In [ ]:
!pip install gnss-lib-py

In [ ]:
!pip install simplekml


## 2. Imports
Import all required libraries for GNSS processing, math, and data handling.

In [4]:
import georinex as gr
import numpy as np
import pandas as pd
import os
import xarray as xr

In [5]:
import gnss_lib_py as glp
from gnss_lib_py.utils.ephemeris_downloader import load_ephemeris
from datetime import datetime, timezone

## 3. Load Data from GitHub
Download the RINEX navigation (`.rnx`) and observation (`.26o`) files directly from the GitHub repository, then load them using `georinex`.

In [ ]:
import urllib.request
import os

# ── GitHub raw base URL ──────────────────────────────────────────────────────
GITHUB_BASE = 'https://raw.githubusercontent.com/amitd55/Ex0-Intro_to_GNSS_navigation/main/rawgnss'

# Download navigation (broadcast ephemeris) file
nav_filename = 'BRDC00IGS_R_20260800000_01D_MN.rnx'
urllib.request.urlretrieve(f'{GITHUB_BASE}/{nav_filename}', nav_filename)
print(f'Downloaded: {nav_filename}')

# Download observation file
obs_filename = 'gnss_log_2026_03_21_17_17_57.26o'
urllib.request.urlretrieve(f'{GITHUB_BASE}/{obs_filename}', obs_filename)
print(f'Downloaded: {obs_filename}')


In [ ]:
import georinex as gr

# Load GPS-only navigation data from downloaded file
nav_file_path = nav_filename

try:
    nav_data = gr.load(nav_file_path, use='G')
    print('Success! GPS nav data loaded.')
except Exception as e:
    print(f'Error loading nav file: {e}')


In [ ]:

obs_file_path = obs_filename


In [ ]:
obs = gr.load(obs_file_path)
print(f'Observation file loaded: {obs_file_path}')


## 4. Constants & Preprocessing
Define physical constants and parse the observation + navigation data into aligned DataFrames indexed by GPS time-of-week (SOW).

### קבועים פיזיקליים

| סמל | ערך | משמעות |
|-----|-----|---------|
| `mu` (μ) | 3.986004418 × 10¹⁴ m³/s² | קבוע כבידה של כדור הארץ (GM) |
| `omega_e` (ωₑ) | 7.2921151467 × 10⁻⁵ rad/s | מהירות סיבוב כדור הארץ |
| `c` | 2.99792458 × 10⁸ m/s | מהירות האור בוואקום |
| `F` | -4.442807633 × 10⁻¹⁰ s/m^½ | קבוע לתיקון יחסותי: F = -2√μ / c² |

גלי רדיו שהלוויינים שולחים נעים כמעט כמהירות האור (בערך 300,000 ק"מ/שנייה).  
**זמן × מהירות = דרך** — כך ממירים זמן טיסה של האות למרחק במטרים.


In [ ]:
mu = 3.986004418e14        # Earth's gravitational constant (m^3/s^2)
omega_e = 7.2921151467e-5  # Earth's rotation rate (rad/s)
c = 2.99792458e8           # Speed of light (m/s)
F = -4.442807633e-10       #Relativistic Correction Constant


### עיבוד מקדים - המרת זמן ל-SOW

**נקודת ה-GPS Epoch:** 6 בינואר 1980.  
זהו ה"מפץ הגדול" של מערכת ה-GPS - כדי שכל הלוויינים והמקלטים בעולם ידברו באותה שפה, הם לא משתמשים בלוח השנה הרגיל אלא סופרים כמה שניות עברו מאז אותו יום.

**המרת זמן אבסולוטי ל-Seconds of Week (SOW):**
```
SOW = (t_GPS - GPS_epoch) % 604800
```
המספר **604,800** = מספר השניות בשבוע שלם (7 × 24 × 3600).  
מערכת ה-GPS עובדת במחזורים של שבוע - בכל יום ראשון בחצות המונה מתאפס ומתחיל לספור מ-0 עד 604,799.

**למה עושים מודולו?**  
כדי להמיר זמן אבסולוטי (מאז 1980) לזמן יחסי בתוך השבוע הנוכחי, כי הנתונים בקובץ ה-NAV מאורגנים לפי SOW.

רק לוויינים מסוג GPS (SV שמתחיל ב-`'G'`) נבחרים מנתוני התצפית.


In [ ]:
gps_epoch = pd.Timestamp('1980-01-06', tz='UTC')

selected_svs = [sv for sv in obs.sv.values if sv.startswith('G') ]

pi_df = obs.sel(sv=selected_svs).to_dataframe().reset_index()
pi_df = pi_df.dropna(subset=['C1C'])

pi_df['time'] = pd.to_datetime(pi_df['time']).dt.tz_localize('UTC')
pi_df['time_seconds'] = (pi_df['time'] - gps_epoch).dt.total_seconds() % 604800

nav_df = nav_data.to_dataframe().dropna(how='all').reset_index()

nav_df['time'] = pd.to_datetime(nav_df['time']).dt.tz_localize('UTC')
nav_df['time_seconds'] = (nav_df['time'] - gps_epoch).dt.total_seconds() % 604800

## 5. Satellite Position Functions
Core GNSS functions:
- **`find_closest_nav`** - finds the nearest ephemeris record for a satellite at a given time
- **`compute_sat_pos`** - computes ECEF satellite position (X, Y, Z) and clock bias from Keplerian orbital elements
- **`process_row`** - applies the above two functions row-wise across the observation DataFrame

### `find_closest_nav` - בחירת הודעת הניווט המתאימה

הפונקציה אחראית על בחירת הודעת הניווט המתאימה ביותר לכל מדידה.

**איך זה עובד:**
1. מסננת את קובץ הניווט עבור לווין ספציפי
2. לוקחת את זמן התצפית ומחסירה ממנו את הזמנים של כל ההודעות עבור הלווין הזה
3. ה-`abs` הופך את כל ההפרשים לחיוביים
4. מבין כל ההפרשים לוקחת את המספר הכי קטן - ההודעה שזמנה הכי קרוב לזמן התצפית
5. מבצעת **בדיקת טריות**: אם ההפרש עולה על **7,200 שניות (שעתיים)** - הנתונים פג תוקפם.  
   שעתיים הוא סטנדרט עולמי של מערכת ה-GPS.


In [ ]:
def find_closest_nav(sv, obs_time_sow, nav_df):
    sv_nav = nav_df[nav_df['sv'] == sv]
    if sv_nav.empty:
        return None

    diffs = (sv_nav['time_seconds'] - obs_time_sow).abs()
    idx = diffs.idxmin()

    if diffs[idx] > 7200:
        return None

    return sv_nav.loc[idx]

### `compute_sat_pos` - חישוב מיקום הלווין ב-ECEF

חישוב X, Y, Z של הלווין מהפרמטרים הקפלריאניים שבקובץ ה-NAV.

---

#### קבוצה 2: בנאי המסלול (Orbital Shape)
הפרמטרים האלו מתארים את האליפסה שבה הלווין נוסע בחלל:

| פרמטר | משמעות |
|--------|---------|
| `sqrtA` | שורש חצי הציר הראשי - מגדיר את **גודל** המסלול |
| `Eccentricity (e)` | אקסצנטריות - כמה המסלול "ביצתי" ולא עיגול מושלם |
| `Toe` | Time of Ephemeris - "זמן הייחוס" שבו הנתונים נמדדו |
| `M0` | אנומליה ממוצעת - איפה הלווין נמצא על המסלול בזמן Toe |

#### קבוצה 3: כיוון המסלול בחלל (Orientation)
| פרמטר | משמעות |
|--------|---------|
| `i0` | נטיית המסלול - הזווית שבה המסלול חותך את קו המשווה |
| `Omega0` | אורך צומת העולה - הזווית שבה המסלול "מסתובב" סביב ציר כדור הארץ |
| `omega` | ארגומנט הפריגיאה - איפה הנקודה הכי קרובה לכדור הארץ בתוך המסלול |

#### קבוצה 4: מתקני "הרעידות" (C-parameters)
Crs, Crc, Cuc, Cus, Cic, Cis - 6 פרמטרים שמתקנים סטיות קטנות ברדיוס, בזווית ובנטייה כי כדור הארץ לא עגול לגמרי. בלי זה תהיה טעות של כמה מטרים.

---

**שלב 1: חישוב הזמן שחלף (tₖ)**
```
tk = time - toe
```
כאשר `time` הוא זמן המדידה מה-obs.

**תיקון גבול השבוע:**
```python
if tk > 302400:  tk -= 604800
if tk < -302400: tk += 604800
```
המספר **302,400** הוא בדיוק חצי שבוע.  
הבעיה: אם המדידה היא בתחילת השבוע (t=2) והנתונים מסוף השבוע הקודם (toe=604,798), חיסור פשוט נותן -604,796 - המחשב יחשוב שעבר כמעט שבוע! במציאות עברו 4 שניות בלבד.  
הפתרון: אם |tk| > חצי שבוע, מוסיפים/מחסירים שבוע שלם כדי לקפל את הזמן למרחב הנכון.

**שלב 2: מהירות ממוצעת מתוקנת (n) ואנומליה ממוצעת (Mₖ)**
```
n  = √(μ / A³) + ΔN
Mₖ = M₀ + n·tₖ
```
Mₖ היא הזווית (ברדיאנים) שבה הלווין היה נמצא לו היה נע במעגל מושלם ובמהירות קבועה.

**שלב 3: משוואת קפלר (Eₖ) - איטרטיבי**
```
Eₖ = Mₖ + e·sin(Eₖ)
```
מתקנים לעובדה שהמסלול הוא אליפסה ולא עיגול.  
בגלל ש-Eₖ מופיע גם כזווית וגם בתוך הסינוס, אי אפשר לפתור "במכה אחת" - משתמשים באיטרציות: מנחשים E = M, בודקים כמה טעינו, ומתקנים שוב ושוב עד שהטעות אפסית.

**שלב 4 (קבוצה 1): תיקון שעון הלווין (dts)**

| פרמטר | משמעות |
|--------|---------|
| `SVclockBias (a0)` | שגיאת השעון של הלוויין |
| `SVclockDrift (a1)` | קצב הסטייה של השעון |
| `SV Clock Drift Rate (a2)` | תאוצת הסטייה |
| `TGD` | Total Group Delay - עיכוב פנימי בציוד הלוויין |

```
d_rel = F · e · √A · sin(Eₖ)          # תיקון יחסותי
dts   = a0 + a1·(t - toc) + d_rel - TGD
```
השעון של הלווין (dts) טועה, אך הטעות ידועה מראש מקובץ ה-NAV.

**שלב 4 (המשך): מיקום 2D במישור המסלול**
```
νₖ   = arctan2(√(1-e²)·sin(Eₖ), cos(Eₖ)-e)   # זווית אמיתית
φₖ   = νₖ + ω                                   # זווית משולבת
```
לפני התיקונים מחברים את זווית הלווין (νₖ) עם זווית המסלול עצמו (ω מקבוצה 3).

**תיקוני "הרעידות" (C-parameters):**
```
uₖ = φₖ + Cus·sin(2φₖ) + Cuc·cos(2φₖ)   # תיקון לזווית
rₖ = A·(1-e·cos(Eₖ)) + Crs·sin(2φₖ) + Crc·cos(2φₖ)   # תיקון לרדיוס
iₖ = i0 + İ·tₖ + Cis·sin(2φₖ) + Cic·cos(2φₖ)   # תיקון לנטייה
```

**שלב 5: לונגיטודה של צומת העולה (Ωₖ) - תיקון סיבוב כדור הארץ**
```
Ωₖ = Ω₀ + (Ω̇ - ωₑ)·tₖ - ωₑ·toe
```

**שלב 6: קואורדינטות ECEF (X, Y, Z)**
```
Xₖ = rₖ · (cos(uₖ)·cos(Ωₖ) - sin(uₖ)·cos(iₖ)·sin(Ωₖ))
Yₖ = rₖ · (cos(uₖ)·sin(Ωₖ) + sin(uₖ)·cos(iₖ)·cos(Ωₖ))
Zₖ = rₖ · sin(uₖ)·sin(iₖ)
```


In [ ]:
def compute_sat_pos(nav_row, obs_time_sow):

    A = nav_row['sqrtA']**2
    e = nav_row['Eccentricity']
    toe = nav_row['Toe']

    tk = obs_time_sow - toe
    if tk > 302400:  tk -= 604800
    if tk < -302400: tk += 604800


    n = np.sqrt(mu / A**3) + nav_row['DeltaN']
    Mk = nav_row['M0'] + n * tk


    Ek = Mk
    for _ in range(10):
        Ek = Mk + e * np.sin(Ek)


    d_rel = F * e * nav_row['sqrtA'] * np.sin(Ek)

    dt_ref = nav_row['time_seconds']
    dts = (nav_row['SVclockBias'] +
           nav_row['SVclockDrift'] * (obs_time_sow - dt_ref) +
           d_rel - nav_row['TGD'])


    vk = np.arctan2(np.sqrt(1 - e**2) * np.sin(Ek), np.cos(Ek) - e)
    phi_k = vk + nav_row['omega']

    uk = phi_k + nav_row['Cus']*np.sin(2*phi_k) + nav_row['Cuc']*np.cos(2*phi_k)
    rk = A*(1 - e*np.cos(Ek)) + nav_row['Crs']*np.sin(2*phi_k) + nav_row['Crc']*np.cos(2*phi_k)
    ik = nav_row['Io'] + nav_row['IDOT']*tk + nav_row['Cis']*np.sin(2*phi_k) + nav_row['Cic']*np.cos(2*phi_k)

    Ok = nav_row['Omega0'] + (nav_row['OmegaDot'] - omega_e)*tk - omega_e*toe

    Xk = rk * (np.cos(uk)*np.cos(Ok) - np.sin(uk)*np.cos(ik)*np.sin(Ok))
    Yk = rk * (np.cos(uk)*np.sin(Ok) + np.sin(uk)*np.cos(ik)*np.cos(Ok))
    Zk = rk * (np.sin(uk)*np.sin(ik))

    return Xk, Yk, Zk, dts

In [ ]:
def process_row(row, nav_df):
    nav_row = find_closest_nav(row['sv'], row['time_seconds'], nav_df)

    if nav_row is not None:
        Xk, Yk, Zk, dts = compute_sat_pos(nav_row, row['time_seconds'])
        return pd.Series([Xk, Yk, Zk, dts], index=['Xk', 'Yk', 'Zk', 'dts'])
    else:
        return pd.Series([None, None, None, None], index=['Xk', 'Yk', 'Zk', 'dts'])

Apply satellite position computation across all observation epochs.

In [ ]:
sat_results = pi_df.apply(process_row, axis=1, args=(nav_df,))
final_df = pd.concat([pi_df, sat_results], axis=1)

## 6. Pseudorange Correction
Correct raw pseudoranges (`C1C`) by applying the satellite clock bias (`dts`), producing clean pseudoranges (`CleanPi`).

### `calculate_clean_pi` — ניקוי ה-Pseudorange

**מהו ה-pi (pseudorange)?**  
- **זמן טיסה** = זמן קבלה − זמן שידור  
- **pi (מרחק מדומה)** = זמן טיסה × מהירות האור — נותן את המרחק במטרים  
- נקרא "מדומה" כי הוא כולל בתוכו את הטעות של השעון במקלט

ה-pi שנתון בקובץ ה-obs הוא **"מלוכלך"** — כולל בתוכו גם את הטעות של השעון שלנו וגם את הטעות של שעון הלווין.  
לכן נרצה לנקות אותו לפני שנתחיל להשתמש בו.

**נוסחת הניקוי:**
```
CleanPi = C1C + (c × dts)
```
- `C1C` — המדידה הגולמית מקובץ ה-obs  
- `dts` — שגיאת שעון הלווין בשניות (שחושבה ב-`compute_sat_pos`)  
- `c × dts` — הפיכת טעות הזמן לטעות במטרים

**השעון שלנו (dt)** הוא הנעלם שנחשב עם 4 המשוואות במטריצה.  
**השעון של הלווין (dts)** — הטעות ידועה מראש מקובץ ה-NAV.

אחרי שמצאנו את dts אנחנו הופכים אותו למטרים ומתקנים את ה-pi.


In [ ]:
def calculate_clean_pi(row):
    if pd.isna(row['Xk']):
        return None

    return row['C1C'] + (c * row['dts'])


In [ ]:
final_df['CleanPi'] = final_df.apply(calculate_clean_pi, axis=1)
final_df = final_df.dropna(subset=['CleanPi'])

## 7. Positioning & Coordinate Utilities
Helper functions for the least-squares solver:
- **`get_elevation_angle`** - filters low-elevation satellites (below mask angle)
- **`least_squares_position`** - iterative weighted least-squares to solve for receiver ECEF position and clock offset
- **`compute_rms`** - computes RMS of pseudorange residuals per epoch
- **`ecef_to_geodetic`** - converts ECEF (X, Y, Z) to geodetic (lat, lon, alt) using pyproj

### `get_elevation_angle` — זווית הגבהה ומסכת הגבהה

**זווית הגבהה** = הזווית שבין קו האופק של המקלט לבין המיקום של הלווין בשמיים.

**מסכת הגבהה** = פילטר שקובע מהי זווית ההגבהה המינימלית שמתחתיה לא מקבלים נתונים מלווין — בקוד היא מוגדרת כ-**5.0 מעלות**.

**למה מסננים לוויינים נמוכים?**
1. **הפרעות באטמוספירה** — האות צריך לעבור דרך שכבה עבה מאוד של האטמוספירה, מה שיוצר עיכובים ושגיאות
2. **חסימות** — בגובה נמוך יש סיכוי שהאות יפגע בבניינים או בעצים ויחזור מהם, מה שמטעה את המקלט לגבי המרחק האמיתי

**חישוב הזווית:**  
הפונקציה ממירה את מיקום המקלט ל-ECEF, מחשבת את וקטור הכיוון לכדור הארץ (`up`), ואז מחשבת את הזווית בין קו הלווין-מקלט לבין כיוון "למעלה".


In [ ]:
def get_elevation_angle(xs, ys, zs, xr, yr, zr):
    lat, lon, _ = ecef_to_geodetic(xr, yr, zr)
    lat_rad, lon_rad = np.radians(lat), np.radians(lon)

    dx, dy, dz = xs - xr, ys - yr, zs - zr
    distance = np.sqrt(dx**2 + dy**2 + dz**2)

    up_x = np.cos(lat_rad) * np.cos(lon_rad)
    up_y = np.cos(lat_rad) * np.sin(lon_rad)
    up_z = np.sin(lat_rad)

    sin_el = (up_x * dx + up_y * dy + up_z * dz) / distance
    return np.degrees(np.arcsin(sin_el))


### `least_squares_position` — פתרון מיקום ב-Least Squares

**הבסיס — זמן × מהירות = דרך:**  
הלווין שולח הודעה עם זמן השידור. המקלט רושם את זמן ההגעה. המרחק = זמן הנסיעה × מהירות האור.  
הבעיה: CleanPi הוא מרחק שקרי כי כולל מרחק אמיתי + טעות שעון.

**4 נעלמים, 4 משוואות (לפחות):**  
נחפש: X, Y, Z (מיקום המקלט) + b (שגיאת שעון המקלט במטרים).  
לכן צריך לפחות **4 לוויינים** בכל אפוקה.

---

**יצירת ניחוש ראשוני:**  
אם הניחוש הוא (0,0,0) — לוקחים את כל הלוויינים הנראים ומחשבים את המיקום הממוצע שלהם.  
ממנרמלים ומכפילים ברדיוס כדור הארץ (6,371 ק"מ) — עוצרים בדיוק בקליפה.  
**למה?** אם מתחילים מ-(0,0,0), המרחק ההתחלתי רחוק 6,371 ק"מ מהפתרון, מה שמאסטבל את המטריצות.

---

**תיקון סיבוב כדור הארץ (אפקט סניאק):**
```
dist_coarse = √(dx² + dy² + dz²)   # מרחק גס בפיתגורס תלת-ממדי
θ = ωₑ × (dist_coarse / c)          # כמה רדיאנים הספיק כדור הארץ להסתובב
Xs' =  Xs·cos(θ) + Ys·sin(θ)       # סיבוב הלווין במישור XY
Ys' = -Xs·sin(θ) + Ys·cos(θ)
```
הקואורדינטות של הלווין הן מרגע השידור, אבל המיקום שלנו הוא מרגע הקליטה — בזמן הזה כדור הארץ הסתובב בזווית θ.  
הסיבוב קורה **רק במישור ה-X וה-Y** — ציר ה-Z לא משתנה כי כדור הארץ מסתובב סביב ציר Z.

---

**בניית המטריצה A ווקטור השאריות b:**

**חישוב כיוון הלווין:**
```
dx = Xs' - X_guess,  dy = Ys' - Y_guess,  dz = Zs - Z_guess
ρ  = √(dx² + dy² + dz²)
```
בודקים כמה רחוק הלווין מהמקלט בכל ציר בנפרד, ואז מחשבים את המרחק האווירי הישיר (אלכסון בפיתגורס).  
מנרמלים כדי לקבל כיוון בלבד (בלי המרחק):
```
ax = dx/ρ,  ay = dy/ρ,  az = dz/ρ
```
הזוויות האלו אומרות לאלגוריתם כיצד לחלק את הטעות בין הצירים.  
ה-1 בסוף אומר למחשב שהשעון (dt) משפיע על הלווין הזה ב-100% בלי קשר לזווית:
```
שורה במטריצה A: [ax, ay, az, 1]
```

**חישוב השארית (b):**
```
b = pr - ρ - guess[3]
```
- `pr` = CleanPi (איפה המכשיר חושב שהוא נמצא — כולל שגיאת שעון)
- `ρ` = המרחק הגיאומטרי בין הניחוש לקואורדינטות הלווין
- `guess[3]` = שגיאת השעון הנוכחית  
**המטרה:** לגרום ל-b להיות הכי קרוב ל-0 עבור כל הלוויינים בבת אחת.

**עדכון הניחוש:**
```
Δx = lstsq(A, b)   # פתרון מינימום ריבועים
guess += Δx
```
חוזרים עד שהשיפור קטן מ-0.1 מטר או אחרי 20 איטרציות.


In [ ]:
def least_squares_position(epoch, guess, elev_mask=5.0, max_iter=20, tol=0.1):


    if np.all(guess[:3] == 0):
        sat_mean = epoch[['Xk', 'Yk', 'Zk']].mean().values
        norm = np.linalg.norm(sat_mean)
        guess[:3] = (sat_mean / norm) * 6371000.0
        first_pr = epoch['CleanPi'].iloc[0]
        dist_to_first = np.linalg.norm(guess[:3] - epoch[['Xk', 'Yk', 'Zk']].iloc[0])
        guess[3] = first_pr - dist_to_first

    for iteration in range(max_iter):
        A, b, used_idx = [], [], []

        for idx, row in epoch.iterrows():
            Xs, Ys, Zs = row['Xk'], row['Yk'], row['Zk']
            pr = row['CleanPi']

            el = get_elevation_angle(Xs, Ys, Zs, guess[0], guess[1], guess[2])
            if el < elev_mask: continue


            dist_coarse = np.sqrt((guess[0]-Xs)**2 + (guess[1]-Ys)**2 + (guess[2]-Zs)**2)
            theta = omega_e * (dist_coarse / c)

            Xs_r = Xs * np.cos(theta) + Ys * np.sin(theta)
            Ys_r = -Xs * np.sin(theta) + Ys * np.cos(theta)

            dx = guess[0] - Xs_r
            dy = guess[1] - Ys_r
            dz = guess[2] - Zs
            rho = np.sqrt(dx**2 + dy**2 + dz**2)

            A.append([dx/rho, dy/rho, dz/rho, 1.0])
            b.append(pr - rho - guess[3])
            used_idx.append(idx)

        if len(A) < 4: break

        A, b = np.array(A), np.array(b)
        delta, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        guess += delta

        if np.linalg.norm(delta[:3]) < tol: break

    return guess, used_idx

### `compute_rms` — שגיאת שורש ממוצע ריבועי

לאחר ההתכנסות, מחשבים את **RMS** של שאריות ה-pseudorange כמדד לאיכות הפתרון:
```
RMS = √( (1/N) · Σ(CleanPiᵢ - ρᵢ - b)² )
```
ה-RMS מצביע כמה מטרים בממוצע "טועה" כל מדידה לאחר הפתרון — ככל שנמוך יותר, הפתרון טוב יותר.

הפונקציה מחשבת את תיקון סיבוב כדור הארץ (סניאק) שוב עבור כל לווין לפני חישוב השארית.


In [ ]:
def compute_rms(epoch, ecef_pos, used_idx):
    residuals = []

    for idx in used_idx:
        row = epoch.loc[idx]

        dx0 = row['Xk'] - ecef_pos[0]
        dy0 = row['Yk'] - ecef_pos[1]
        dz0 = row['Zk'] - ecef_pos[2]
        dist_coarse = np.sqrt(dx0**2 + dy0**2 + dz0**2)
        theta = omega_e * (dist_coarse / c)

        Xs_r = row['Xk'] * np.cos(theta) + row['Yk'] * np.sin(theta)
        Ys_r = -row['Xk'] * np.sin(theta) + row['Yk'] * np.cos(theta)

        dx = ecef_pos[0] - Xs_r
        dy = ecef_pos[1] - Ys_r
        dz = ecef_pos[2] - row['Zk']
        rho = np.sqrt(dx**2 + dy**2 + dz**2)

        res = row['CleanPi'] - (rho + ecef_pos[3])
        residuals.append(res)

    if len(residuals) == 0:
        return None, 0

    residuals = np.array(residuals)
    rms = np.sqrt(np.mean(residuals**2))
    return rms, len(residuals)

In [ ]:
from pyproj import Transformer
from pyproj import CRS
crs = CRS.from_epsg(4326)

def ecef_to_geodetic(x,y,z):
  transformer = Transformer.from_crs("EPSG:4978", "EPSG:4326", always_xy=True)
  lon, lat, alt = transformer.transform(x, y, z)

  return lat, lon, alt


## 8. Trajectory Computation
Main processing loop - iterates over each 1 Hz epoch, applies satellite position refinement using signal travel time, runs two-pass least-squares positioning, and collects valid fixes.

### לולאת חישוב המסלול — שני מעברי Least Squares

**מדוע שני מעברים?**  
הלווין הוא מטרה נעה במהירות גבוהה (~4 ק"מ/שנייה). בזמן שהאות טס (בערך 0.07 שניות) הלווין מתקדם כמעט **300 מטרים** במסלול שלו.

---

**מעבר ראשון — ניחוש גס:**  
מריצים Least Squares עם קואורדינטות הלווין מרגע הקליטה (לא מדויק אבל מספיק לניחוש).  
`res_p1[3]` = שגיאת השעון במטרים → מחלקים ב-c → שגיאת השעון בשניות (`dt_seconds`).

**תיקון זמן השידור:**
```python
tau    = C1C / c            # זמן הטיסה של האות (שניות)
tx_time = t - dt_seconds - tau   # הזמן בו הלווין שלח את האות
```
- `C1C` — המדידה הגולמית: בדיוק מהירות האור × ההפרש בין שעון הקליטה לשעון השידור.  משמש כ"סרגל הזמן"
- `CleanPi` — C1C אחרי תיקון שעון הלווין. משמש כ"סרגל המרחק"

ניגשים לקובץ ה-NAV שוב — **הפעם עם זמן השידור** ולא זמן הקליטה.  
`compute_sat_pos` מחשבת איפה הלווין היה **באמת** ברגע השידור.

**מעבר שני — פתרון מדויק:**  
מריצים Least Squares עם הקואורדינטות המתוקנות של רגע השידור.

---

**סינון פיזיקלי:**
```
-100 < alt < 3000
```
- 3000 מ': שיא החרמון הוא בערך 2,814 מ'
- -100 מ': כל עוד לא עמוק בתוך האדמה או בים המלח (-430 מ')  
מסנן אאוטליירים שנוצרים משגיאות מתמטיות בתהליך ההתכנסות.


In [ ]:
trajectory_results = []
current_guess = np.array([0.0, 0.0, 0.0, 0.0])

for timestamp_sow, epoch in final_df.groupby('time_seconds'):
    clean_epoch = epoch.dropna(subset=['Xk', 'Yk', 'Zk', 'CleanPi']).copy()
    if len(clean_epoch) < 4: continue

    res_p1, used_idx_p1 = least_squares_position(clean_epoch, current_guess.copy())
    if res_p1 is None: continue

    dt_seconds = res_p1[3] / c

    for idx, row in clean_epoch.iterrows():
        tau = row['C1C'] / c
        tx_time = timestamp_sow - dt_seconds - tau

        nav_row = find_closest_nav(row['sv'], tx_time, nav_df)
        if nav_row is not None:
            xk, yk, zk, dts = compute_sat_pos(nav_row, tx_time)
            clean_epoch.at[idx, 'Xk'] = xk
            clean_epoch.at[idx, 'Yk'] = yk
            clean_epoch.at[idx, 'Zk'] = zk
            clean_epoch.at[idx, 'CleanPi'] = row['C1C'] + (c * dts)

    final_res, used_idx_p2 = least_squares_position(clean_epoch, res_p1)
    if final_res is None: continue

    current_guess = final_res

    rms_value, n_clean = compute_rms(clean_epoch, final_res, used_idx_p2)
    lat, lon, alt = ecef_to_geodetic(final_res[0], final_res[1], final_res[2])

    if -100 < alt < 3000:
        trajectory_results.append({'time_sow': timestamp_sow, 'lat': lat, 'lon': lon, 'alt': alt, 'rms': rms_value, 'sat_count': n_clean})

final_path_df = pd.DataFrame(trajectory_results)

Preview the resulting trajectory DataFrame.

In [ ]:
final_path_df

## 9. Export Results
Save the computed trajectory as a **CSV** file and a **KML** file for visualization in Google Earth.
Output files can be downloaded from the Colab file browser (📁 icon on the left sidebar).

In [ ]:
# Save trajectory to CSV 
output_csv = 'final_path_df.csv'
final_path_df.to_csv(output_csv, index=False)
print(f'Saved: {output_csv}')


In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(1, 1, figsize=(8, 6)) # Create only one subplot

ax1.scatter(final_path_df['lon'], final_path_df['lat'], c='red', s=10)
ax1.set_title("Map View (Lat vs Lon)")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
ax1.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import simplekml

# Save KML locally (download from Colab file browser on the left)
def export_to_kml(df, filename="my_trajectory.kml"):

    kml = simplekml.Kml()

    point_folder = kml.newfolder(name="Trajectory Points")

    for idx, row in df.iterrows():
        pnt = point_folder.newpoint(name="")
        pnt.coords = [(row['lon'], row['lat'], row['alt'])]

        pnt.style.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/pushpin/red-pushpin.png'
        pnt.style.iconstyle.scale = 0.8

        pnt.description = ""

        pnt.altitudemode = simplekml.AltitudeMode.clamptoground
        pnt.extrude = 0

    line_folder = kml.newfolder(name="Trajectory Line")
    ls = line_folder.newlinestring(name="Full Trajectory")
    ls.coords = [(row['lon'], row['lat'], row['alt']) for index, row in df.iterrows()]

    ls.altitudemode = simplekml.AltitudeMode.clamptoground
    ls.extrude = 0
    ls.tessellate = 1

    ls.style.linestyle.color = simplekml.Color.red
    ls.style.linestyle.width = 3

    kml.save(filename)
    print(f"Success! KML file saved as: {filename}")

export_to_kml(final_path_df)